## Text Extraction, Cleaning, and Rephrasing from multiple books

In [1]:
import pandas as pd
import traceback
import json
from PyPDF2 import PdfReader
import os

from bson import ObjectId
from motor.motor_asyncio import AsyncIOMotorClient
from langchain_openai import ChatOpenAI
from langchain import PromptTemplate, LLMChain

from latext_Prompts import (
EXTRACTION_SYSTEM,
EXTRACTION_USER,
EXTRACTION_OUTPUT_FORMAT,
MATHEMATICS_EXTRACTION_OUTPUT_EXAMPLE,
BIOLOGY_EXTRACTION_OUTPUT_EXAMPLE,
PHYSICS_EXTRACTION_OUTPUT_EXAMPLE,
CHEMISTRY_EXTRACTION_OUTPUT_EXAMPLE,
EMPTY_EXTRACTION_OUTPUT_FORMAT,
CLEAN_REPHRASE_USER,
CLEAN_REPHRASE_SYSTEM,
REPHRASE_OUTPUT_FORMAT,
REPHRASE_OUTPUT_EXAMPLE,
BIOLOGY_SOLUTION_SYSTEM_PROMPT,
CHEMISTRY_SOLUTION_SYSTEM_PROMPT,
MATHEMATICS_SOLUTION_SYSTEM_PROMPT,
PHYSICS_SOLUTION_SYSTEM_PROMPT,
DISTRACTOR_SYSTEM_PROMPT
)

from config import (
    OPENAI_API_KEY,
    MONGO_URL,
    DB_NAME
)

In [2]:
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "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"
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_PROJECT"] = "Questions Extractor"

# Pipeline

In [2]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [3]:
# All question collection

def collect_questions_from_chapter_with_Langchain(subject, chapter_text, chapter_name, grade):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
        )
        match subject:
            case "mathematics":
                examples=MATHEMATICS_EXTRACTION_OUTPUT_EXAMPLE
            case "biology":
                examples=BIOLOGY_EXTRACTION_OUTPUT_EXAMPLE
            case "physics":
                examples =PHYSICS_EXTRACTION_OUTPUT_EXAMPLE
            case "chemistry":
                examples=CHEMISTRY_EXTRACTION_OUTPUT_EXAMPLE

        #formatting prompts
        system = EXTRACTION_SYSTEM.format("",grade = grade, subject = subject,EMPTY_EXTRACTION_OUTPUT_FORMAT = EMPTY_EXTRACTION_OUTPUT_FORMAT, out_format = EXTRACTION_OUTPUT_FORMAT, out_example = examples)
        user = EXTRACTION_USER.format(chapter_name = chapter_name, chapter_text = chapter_text)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ],
        )
        return response
    except Exception as e:
        print("Error with in extraction: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return None

In [4]:
def clean_rephrase_question(question_text, topic_list):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
            # verbose = True
        )

        #formatting prompts
        system = CLEAN_REPHRASE_SYSTEM.format("",out_example = REPHRASE_OUTPUT_EXAMPLE, out_format = REPHRASE_OUTPUT_FORMAT)
        user = CLEAN_REPHRASE_USER.format(question_text = question_text, topic_list = topic_list)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ]
        )
        return response
    except Exception as e:
        print("Error cleaning text: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return question_text  # Return original text in case of an error

In [5]:
import os
import traceback
from pathlib import Path
import zipfile
import tempfile
import shutil

def read_data_from_latex(publication: str, chapter_name: str,grade:str) -> str:
    """
    Read LaTeX content from a specified publication and chapter zip file.
    
    Args:
        publication (str): Name of the publication (e.g., 'mtg')
        chapter_name (str): Name of the chapter (e.g., 'Life Processes')
    
    Returns:
        str: Content of the LaTeX file if successful, empty string if failed
    """
    try:
        # Construct the zip file path
        zip_path = os.path.join(r"textbooks", publication, f"{chapter_name}_{grade}.zip")
        
        # Verify zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Create a temporary directory to extract files
        with tempfile.TemporaryDirectory() as temp_dir:
            # Extract the zip file
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(temp_dir)
            
            # Find directories starting with 2025
            content_dirs = [d for d in os.listdir(temp_dir) 
                          if os.path.isdir(os.path.join(temp_dir, d)) and 
                          d.startswith('2025')]
            
            if not content_dirs:
                raise FileNotFoundError(f"No content directories found in zip file")
                
            # Get the latest directory
            latest_dir = sorted(content_dirs)[-1]
            dir_path = os.path.join(temp_dir, latest_dir)
            
            # Find the .tex file in the directory
            tex_files = [f for f in os.listdir(dir_path) if f.endswith('.tex')]
                    
            if not tex_files:
                raise FileNotFoundError(f"No .tex file found in {dir_path}")
                
            # Full path to the tex file
            latex_path = os.path.join(dir_path, tex_files[0])
            
            # Read the content
            with open(latex_path, 'r', encoding='utf-8') as file:
                content = file.read()
                
            print(f"Successfully processed LaTeX file from zip: {latex_path}")
            return content

    except Exception as e:
        print(
            "Failed to process the LaTeX file. Exception Occurred:",
            type(e).__name__,
            "–",
            e,
            "\n",
            traceback.format_exc()
        )
        return ""

In [6]:
def extract_questions(subject: str, chapter_texts: list, chapter_name: str, grade: str) -> list:
    try:
        # Validate inputs
        if not isinstance(chapter_texts, list):
            raise ValueError("chapter_texts must be a list of strings")
            
        if not chapter_texts:
            print(f"Warning: Empty chapter_texts for chapter {chapter_name}")
            return []
            
        all_questions = []
        
        # Process lines in chunks of 500
        chunk_size = 250
        for chunk_index in range(0, len(chapter_texts), chunk_size):
            try:
                # Get current chunk of lines
                chunk = chapter_texts[chunk_index:chunk_index + chunk_size]
                
                # Skip empty chunks
                if not chunk:
                    continue
                    
                print(f"Processing chunk {chunk_index//chunk_size + 1}/{-(-len(chapter_texts)//chunk_size)}")
                
                # Use the original collection function
                questions = collect_questions_from_chapter_with_Langchain(
                    subject=subject,
                    chapter_text="\n".join(chunk),
                    chapter_name=chapter_name,
                    grade=grade
                )
                
                # Parse and store valid questions
                if questions and questions.content:
                    parsed = parse_Response(questions.content)
                    if parsed and "questions" in parsed:
                        questions_list = parsed["questions"]
                        if questions_list:
                            all_questions.extend(questions_list)
                            print(f"Extracted {len(questions_list)} questions from chunk {chunk_index//chunk_size + 1}")
                
            except Exception as e:
                print(f"Error processing chunk {chunk_index//chunk_size + 1}: {str(e)}")
                continue
        
        print(f"Total questions extracted: {len(all_questions)}")
        return all_questions

    except Exception as e:
        print(f"Error in extract_questions: {str(e)}")
        return []

In [7]:
def rephrase_questions(conf_data: dict, all_questions: list, topics: list) -> dict:
    print("Starting the cleanup and rephrasing process...")
    try:
        # Validate inputs
        if not conf_data or "chapter_name" not in conf_data:
            raise ValueError("Missing chapter_name in configuration")
            
        if not topics:
            print("\n\nNo topics found in the data.")
            raise ValueError("No topics found in the data")
            
        chapter = conf_data["chapter_name"]
        clean_rephrased_questions = {}
        chapterwise_formatted_questions = []
        
        if not all_questions:
            raise ValueError(f"No questions found for chapter {chapter}")
            
        print(f"\nWorking on chapter {chapter}")
        print(f"Total questions to process: {len(all_questions)}")
        
        # Process questions in batches of 5
        batch_size = 5
        for batch_start in range(0, len(all_questions), batch_size):
            try:
                # Get current batch of questions
                question_batch = all_questions[batch_start:batch_start + batch_size]
                print(f"\nProcessing batch {batch_start//batch_size + 1}/{-(-len(all_questions)//batch_size)}")
                
                # Clean and rephrase the batch
                cleaned_rephrased_questions = clean_rephrase_question(question_batch, topics)
                print("Raw response:", cleaned_rephrased_questions)
                
                # Parse the cleaned response
                if cleaned_rephrased_questions and cleaned_rephrased_questions.content:
                    formatted_questions = parse_Response(cleaned_rephrased_questions.content)
                    if formatted_questions and "questions" in formatted_questions:
                        chapterwise_formatted_questions.extend(formatted_questions["questions"])
                        print(f"Successfully processed {len(formatted_questions['questions'])} questions in batch")
                    else:
                        print(f"Warning: No valid questions found in batch {batch_start//batch_size + 1}")
                
                print("-" * 100)
                
            except Exception as batch_error:
                print(f"Error processing batch {batch_start//batch_size + 1}: {str(batch_error)}")
                continue
        
        # Store processed questions if any were successful
        if chapterwise_formatted_questions:
            clean_rephrased_questions[chapter] = chapterwise_formatted_questions
            print(f"\nSuccessfully processed {len(chapterwise_formatted_questions)} questions for chapter {chapter}")
        else:
            print(f"\nWarning: No questions were successfully processed for chapter {chapter}")
        
        return clean_rephrased_questions
        
    except Exception as e:
        print(f"Failed to clean and rephrase questions. Exception Occurred: {type(e).__name__} – {str(e)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return {}

In [8]:
from typing import Tuple, Dict, Optional
import traceback
import logging

def extract_rephrase_questions(conf_data: dict, topics: list) -> Tuple[Optional[Dict], Optional[Dict]]:
    try:
        # Validate configuration data
        required_fields = ["subject", "grade", "chapter_name"]
        missing_fields = [field for field in required_fields if field not in conf_data]
        if missing_fields:
            raise ValueError(f"Missing required configuration fields: {', '.join(missing_fields)}")
            
        if not topics:
            raise ValueError("No topics provided for question rephrasing")
            
        print("Starting question extraction and rephrasing pipeline...")
        
        # Step 1: Scrape text from PDF/LaTeX
        print("\nStep 1: Scraping textbook content...")
        pdf_text = read_data_from_latex(publication=conf_data['publication'] , chapter_name=conf_data['chapter_name'] , grade=conf_data['grade'])
        if not pdf_text:
            raise ValueError("No text content extracted from textbook")
        print(f"Successfully extracted {len(pdf_text)} text segments")
            
        # Step 2: Extract questions from text
        print("\nStep 2: Extracting questions from text...")
        extracted_questions = extract_questions(
            subject=conf_data["subject"],
            grade=conf_data["grade"],
            chapter_name=conf_data["chapter_name"],
            chapter_texts=pdf_text.split("\n")
        )
        if not extracted_questions:
            raise ValueError("No questions were extracted from the text")
        print(f"Successfully extracted questions")
            
        # Step 3: Clean and rephrase questions
        print("\nStep 3: Cleaning and rephrasing questions...")
        final_rephrased_questions = rephrase_questions(
            conf_data=conf_data,
            all_questions=extracted_questions,
            topics=topics
        )
        if not final_rephrased_questions:
            raise ValueError("No questions were successfully rephrased")
        print("Successfully rephrased questions")
            
        # Return results
        print("\nPipeline completed successfully!")
        return extracted_questions, final_rephrased_questions
        
    except ValueError as val_err:
        print(f"Validation error: {str(val_err)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    except Exception as e:
        print(f"Unexpected error in question processing pipeline: {str(e)}")
        print(f"Exception type: {type(e).__name__}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    finally:
        print("\nQuestion processing pipeline finished")

In [9]:
# get chapter and book data from defaultConf
with open("defaultConf.json", "r") as f:
    conf_data = json.load(f)

chapter = conf_data["chapter_name"]
subject = conf_data["subject"]


print("Chapter to work on: ", chapter)
print(subject)
print(conf_data['publication'])
print(conf_data['grade'])

Chapter to work on:  Physical Quantities and Measurement
physics
goyal
6


In [10]:
import pandas as pd

def process_filename(filename):
    """
    Process filename by stripping whitespace and converting to lowercase
    """
    return filename.strip().lower()
grade = conf_data['grade']
# Match subject to determine sheet name
match subject:
    case "mathematics":
        sheet_name = f"G{grade} Maths"
    case "biology":
        sheet_name = f"G{grade} Bio"
    case "physics":
        sheet_name = f"G{grade} Phy"
    case "chemistry":
        sheet_name = f"G{grade} Chem"
print(sheet_name)
# Read Excel file and process the filename
file_name = process_filename("LEAP ICSE DEFAULTs.xlsx")
misconceptions_df = pd.read_excel(file_name, sheet_name=sheet_name)

# Clean column names
misconceptions_df.columns = misconceptions_df.columns.str.strip()
misconceptions_df.fillna("", inplace=True)

# Process chapter name from conf_data for comparison
chapter_name = conf_data["chapter_name"].strip().lower()

# Get unique topics for the specified chapter
tempTopics = list(misconceptions_df[
    misconceptions_df["Chapter Name"].str.strip().str.lower() == chapter_name
]["Topic"].unique())

# Create topics dictionary with processed IDs
topics = {}
for i in range(len(tempTopics)):
    topic_id = f"{conf_data['grade']}_{conf_data['subject']}_{chapter_name}_{i+1}"
    topics[topic_id] = tempTopics[i].strip()

print("Topics with ID: ", topics, "\n\n", topics.values())

# Process misconceptions and LUs
total_misconceptions = {}
total_LUs = {}

for i, row in misconceptions_df.iterrows():
    topic = str(row["Topic"]).strip()
    
    if topic in topics.values():
        # Process LUs
        if topic in total_LUs:
            total_LUs[topic].append(row["Sub-topic/LU"])
        else:
            total_LUs[topic] = [row["Sub-topic/LU"]]

        # # Process misconceptions
        # temp_misconceptions = [
        #     row[f"Misconceptions {i}"] 
        #     for i in range(1, 11) 
        #     if row[f"Misconceptions {i}"] != ""
        # ]

        # if topic in total_misconceptions:
        #     total_misconceptions[topic].extend(temp_misconceptions)
        # else:
        #     total_misconceptions[topic] = temp_misconceptions

print(total_LUs, sep="\n\n-------------------------------------------------------------------\n\n")

G6 Phy
Topics with ID:  {'6_physics_physical quantities and measurement_1': 'Measurement of Length', '6_physics_physical quantities and measurement_2': 'Measurement of Mass', '6_physics_physical quantities and measurement_3': 'Measurement of Time', '6_physics_physical quantities and measurement_4': 'Temperature and its Measurement', '6_physics_physical quantities and measurement_5': 'Types of Thermometers', '6_physics_physical quantities and measurement_6': 'Measurement of Area'} 

 dict_values(['Measurement of Length', 'Measurement of Mass', 'Measurement of Time', 'Temperature and its Measurement', 'Types of Thermometers', 'Measurement of Area'])
{'Measurement of Length': ['Concept of length as distance between two points.', 'Measurement of length using a ruler and measuring tape', 'Units of length'], 'Measurement of Mass': ['Properties of material- Mass', 'Measurement of mass using beam balance and electronic balance', 'Units of mass'], 'Measurement of Time': ['Measurement of time us

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\64891827.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  misconceptions_df.fillna("", inplace=True)


In [16]:
total_LUs

{'Measurement of Length': ['Concept of length as distance between two points.',
  'Measurement of length using a ruler and measuring tape',
  'Units of length'],
 'Measurement of Mass': ['Properties of material- Mass',
  'Measurement of mass using beam balance and electronic balance',
  'Units of mass'],
 'Measurement of Time': ['Measurement of time using clock, watch and stop watch',
  'Units of Time'],
 'Temperature and its Measurement': ['Sense of touch as temperature sensor',
  'Temperature, Its Measurement Scales, and Conversion Methods'],
 'Types of Thermometers': ['Temperature Measurement of Human Body using clinical thermometer',
  'Temperature Measurement of Human Body using digital clinical thermometer',
  'Correct Way of using Laboratory thermometer'],
 'Measurement of Area': ['Concept of area and its units',
  'Measurement of area of objects of regular shapes']}

In [12]:
extracted_raw_questions_json ,clean_rephrased_questions_json = extract_rephrase_questions(conf_data, topics)


Starting question extraction and rephrasing pipeline...

Step 1: Scraping textbook content...
Successfully processed LaTeX file from zip: C:\Users\ANIKET~1\AppData\Local\Temp\tmpo0yyhpwi\2025_01_28_f063d28aa35ccb17e8cag\2025_01_28_f063d28aa35ccb17e8cag.tex
Successfully extracted 76636 text segments

Step 2: Extracting questions from text...
Processing chunk 1/6
Processing chunk 2/6
Error decoding JSON response: JSONDecodeError - Invalid \escape: line 1 column 571 (char 570)

 Still trying to work on particular exceptions ...
Before removing \ issue:  {
    "questions": [
        {
            "question": "\begin{aligned} \text{Imagine you are asked to find the diameter of thin copper wire by using a centimetre scale. If you directly use the centimetre scale you will notice that the diameter of wire is less than 1 mm. In order to find diameter accurately, wind the wire over a smooth pencil in the form of a tight coil, such that there are 30 turns in the coil as shown in Fig. 2.9. Place 

In [13]:
print("Chapter name: ",chapter, ",  Raw questions: ",  len(extracted_raw_questions_json))
print("Chapter name: ",chapter, ",  Cleaned questions: ",  len(clean_rephrased_questions_json[chapter]))

Chapter name:  Physical Quantities and Measurement ,  Raw questions:  124
Chapter name:  Physical Quantities and Measurement ,  Cleaned questions:  121


In [14]:
extracted_raw_questions_json

[{'question': '\\begin{aligned} \\text{Imagine you are asked to find the diameter of thin copper wire by using a centimetre scale. If you directly use the centimetre scale you will notice that the diameter of wire is less than 1 mm. In order to find diameter accurately, wind the wire over a smooth pencil in the form of a tight coil, such that there are 30 turns in the coil as shown in Fig. 2.9. Place the pencil along with the coil along the edge of centimetre scale. If the length of coil is 1.5 cm, then the diameter of wire is} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{What is length? Name four early units of measurement of length.} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{Why do we need standard units for measurement?} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{What are standard international units of measurement? State the unit of length in this system.} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{Define the term metre. Name o

In [15]:
# extracted_raw_questions_json
for json in clean_rephrased_questions_json[conf_data['chapter_name']]:
    print(json['question'])

\begin{aligned} &\text{If a thin copper wire is wound around a pencil to form a coil with 25 turns, and the length of the coil is measured to be 2 cm, what is the diameter of the wire?} \end{aligned}
\begin{aligned} &\text{Which of the following were early units of measurement for length?} \end{aligned}
\begin{aligned} &\text{Why is it important to have standard units for measurement?} \end{aligned}
\begin{aligned} &\text{What are the standard international units of measurement, and what is the unit of length in this system?} \end{aligned}
\begin{aligned} &\text{How is the term metre defined, and what are one multiple and two submultiples of metre?} \end{aligned}
\begin{aligned} &\text{Which position of the eye records the accurate length of a pencil, and what is the magnitude of this length?} \end{aligned}
\begin{aligned} &\text{What are two characteristics of a standard unit for measuring a physical quantity?} \end{aligned}
\begin{aligned} &\text{How can you determine the length of a

# Solution and Distractors(Based on misconceptions)

In [17]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.2, api_key = OPENAI_API_KEY ,  request_timeout=30.0)

In [18]:
questions_df = pd.DataFrame(clean_rephrased_questions_json[conf_data["chapter_name"]])
questions_df

,topic,topic_id,question
0,Measurement of Length,6_physics_physical quantities and measurement_1,\begin{aligned} &\text{If a thin copper wire i...
1,Measurement of Length,6_physics_physical quantities and measurement_1,\begin{aligned} &\text{Which of the following ...
2,Measurement of Length,6_physics_physical quantities and measurement_1,\begin{aligned} &\text{Why is it important to ...
3,Measurement of Length,6_physics_physical quantities and measurement_1,\begin{aligned} &\text{What are the standard i...
4,Measurement of Length,6_physics_physical quantities and measurement_1,\begin{aligned} &\text{How is the term metre d...
...,...,...,...
116,Measurement of Time,6_physics_physical quantities and measurement_3,\begin{aligned} &\text{How many decades are th...
117,Measurement of Time,6_physics_physical quantities and measurement_3,\begin{aligned} &\text{How many seconds are th...
118,Types of Thermometers,6_physics_physical quantities and measurement_5,\begin{aligned} &\text{Why is a clinical therm...
119,Temperature and its Measurement,6_physics_physical quantities and measurement_4,\begin{aligned} &\text{What can be inferred if...


In [19]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [20]:
## Generating solutions
prompt_template = PromptTemplate(
    input_variables=["system_prompt", "question", "topic", "topic_id", "lulist"],
    template="{system_prompt}\n\n\nGenerate a hint, and solution for the given question. Also rephrase the given question according to the specified steps.\nHere are the required context:\nQuestion: {question}\nLearning unit data which defines scope from which solution should be generated: {lulist}\nTopic: {topic}\nTopic ID: {topic_id}\n\nReturn the output in the specified JSON format."
)

chain = LLMChain(llm=llm, prompt=prompt_template)
# chain = prompt_template | llm
# Function to process each question and generate hint, solution, and final answer
def generate_explanation(subject, question, topic, topic_id):
    try:
        match subject:
            case "mathematics":
                system = MATHEMATICS_SOLUTION_SYSTEM_PROMPT
            case "biology":
                system = BIOLOGY_SOLUTION_SYSTEM_PROMPT
            case "physics":
                system = PHYSICS_SOLUTION_SYSTEM_PROMPT
            case "chemistry":
                system = CHEMISTRY_SOLUTION_SYSTEM_PROMPT

        response = chain.run(
            system_prompt=system,
            lulist=total_LUs,
            question=question,
            topic=topic,
            topic_id=topic_id
        )
        return response
    except Exception as e:
        print(f"Error processing question: {question}\nError: {str(e)}")
        return None
# parser = PydanticOutputParser(pydantic_object=sol_data)
# global explanations
explanations = []
def loopForSolution(subject, questions_df):
    counter = 0
    retries_counter = 0
    maxCounter = len(questions_df)
    while counter < maxCounter:
        row = questions_df.iloc[counter]
        question = row['question']
        topic = row['topic']
        topic_id = row['topic_id']
        print(f"Processing question {counter + 1} out of {len(questions_df)}")
        explanation = generate_explanation(subject, question, topic, topic_id)
        parsed_explanation = parse_Response(explanation)
        if parsed_explanation:
            parsed_explanation["question"] = question
            parsed_explanation["topic"] = topic
            parsed_explanation["topic_id"] = topic_id
            explanations.append(parsed_explanation)
            counter += 1
            retries_counter = 0
        else:
            print("Issue in openai response", explanation, "\n\n Inputs: \n\n", question, "\n\n Topic: ", topic)
            print(f"Failed to parse explanation or generate solution for question {counter + 1}, retrying...")
            retries_counter += 1
        if retries_counter == 3:
            counter += 1

loopForSolution(subject, questions_df)

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\1295566236.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_3636\1295566236.py:22: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chain.run(


Processing question 1 out of 121
Processing question 2 out of 121
Processing question 3 out of 121
Processing question 4 out of 121
Processing question 5 out of 121
Processing question 6 out of 121
Processing question 7 out of 121
Processing question 8 out of 121
Processing question 9 out of 121
Processing question 10 out of 121
Processing question 11 out of 121
Processing question 12 out of 121
Processing question 13 out of 121
Processing question 14 out of 121
Processing question 15 out of 121
Processing question 16 out of 121
Processing question 17 out of 121
Processing question 18 out of 121
Processing question 19 out of 121
Processing question 20 out of 121
Processing question 21 out of 121
Processing question 22 out of 121
Processing question 23 out of 121
Processing question 24 out of 121
Processing question 25 out of 121
Processing question 26 out of 121
Processing question 27 out of 121
Processing question 28 out of 121
Processing question 29 out of 121
Processing question 30 

In [21]:
for i in explanations:
    print(i['question'])
    print(i['solution'])
    print("-------------")

\begin{aligned} &\text{If a thin copper wire is wound around a pencil to form a coil with 25 turns, and the length of the coil is measured to be 2 cm, what is the diameter of the wire?} \end{aligned}
\begin{aligned} &\text{The length of the coil is given as 2 cm, and there are 25 turns.} \\ &\text{The length of the coil is equal to the number of turns multiplied by the circumference of the wire.} \\ &\text{Let } d \text{ be the diameter of the wire. The circumference of the wire is } \pi d. \\ &\text{Thus, the total length of the coil is } 25 \times \pi d = 2 \text{ cm.} \\ &\text{Solving for } d, \text{ we have:} \\ &\pi d = \frac{2}{25} \\ &d = \frac{2}{25 \pi} \\ &\text{Therefore, the diameter of the wire is } \frac{2}{25 \pi} \text{ cm.} \\ \end{aligned}
-------------
\begin{aligned} &\text{Which of the following were early units of measurement for length?} \end{aligned}
\begin{aligned} &\text{In ancient times, people used various units of measurement for length based on familiar o

### Generating Distractors

In [22]:
data = pd.DataFrame(explanations)


In [23]:
data['misconception_options'] = ''
data

,hint,solution,question,topic,topic_id,misconception_options
0,\begin{aligned} &\text{To find the diameter of...,\begin{aligned} &\text{The length of the coil ...,\begin{aligned} &\text{If a thin copper wire i...,Measurement of Length,6_physics_physical quantities and measurement_1,
1,\begin{aligned} &\text{Think about the histori...,"\begin{aligned} &\text{In ancient times, peopl...",\begin{aligned} &\text{Which of the following ...,Measurement of Length,6_physics_physical quantities and measurement_1,
2,\begin{aligned} &\text{Consider how having a c...,\begin{aligned} &\text{Standard units of measu...,\begin{aligned} &\text{Why is it important to ...,Measurement of Length,6_physics_physical quantities and measurement_1,
3,\begin{aligned} &\text{The International Syste...,\begin{aligned} &\text{The International Syste...,\begin{aligned} &\text{What are the standard i...,Measurement of Length,6_physics_physical quantities and measurement_1,
4,\begin{aligned} &\text{Think about the definit...,\begin{aligned} &\text{The metre is defined as...,\begin{aligned} &\text{How is the term metre d...,Measurement of Length,6_physics_physical quantities and measurement_1,
...,...,...,...,...,...,...
116,\begin{aligned} &\text{To find the number of d...,\begin{aligned} &\text{A decade consists of 10...,\begin{aligned} &\text{How many decades are th...,Measurement of Time,6_physics_physical quantities and measurement_3,
117,\begin{aligned} &\text{To find the total secon...,\begin{aligned} &\text{To calculate the number...,\begin{aligned} &\text{How many seconds are th...,Measurement of Time,6_physics_physical quantities and measurement_3,
118,\begin{aligned} &\text{Consider the normal ran...,\begin{aligned} &\text{A clinical thermometer ...,\begin{aligned} &\text{Why is a clinical therm...,Types of Thermometers,6_physics_physical quantities and measurement_5,
119,\begin{aligned} &\text{Consider the normal bod...,\begin{aligned} &\text{A normal human body tem...,\begin{aligned} &\text{What can be inferred if...,Temperature and its Measurement,6_physics_physical quantities and measurement_4,


In [24]:
for i in range(len(data)):
    print(data['solution'][i].split("\n"))

['\\begin{aligned} &\\text{The length of the coil is given as 2 cm, and there are 25 turns.} \\\\ &\\text{The length of the coil is equal to the number of turns multiplied by the circumference of the wire.} \\\\ &\\text{Let } d \\text{ be the diameter of the wire. The circumference of the wire is } \\pi d. \\\\ &\\text{Thus, the total length of the coil is } 25 \\times \\pi d = 2 \\text{ cm.} \\\\ &\\text{Solving for } d, \\text{ we have:} \\\\ &\\pi d = \\frac{2}{25} \\\\ &d = \\frac{2}{25 \\pi} \\\\ &\\text{Therefore, the diameter of the wire is } \\frac{2}{25 \\pi} \\text{ cm.} \\\\ \\end{aligned}']
['\\begin{aligned} &\\text{In ancient times, people used various units of measurement for length based on familiar objects or body parts.} \\\\ &\\text{Some of these early units included the cubit, which was the length of the forearm from the elbow to the tip of the middle finger, and the foot, which was based on the length of a human foot.} \\\\ &\\text{These units were not standardized

In [27]:
# Define the prompt template for generating misconceptions
prompt_template = PromptTemplate(
    input_variables=["system_prompt","topic", "topic_id", "question", "hint", "solution"],
    template="{system_prompt}\n\nGenerate one correct option and appropriate incorrect options in latex using given solution, hint, and misconceptions for the given question.\n\nTopic ID: {topic_id}\nTopic: {topic}\nQuestion: {question}\nHint: {hint}\nSolution: {solution}\n\nReturn the output in the specified JSON format."
)

# Extract relevant information from the 'explanation' column

def extract_explanation_details(explanation):
    try:
        explanation_data = json.loads(explanation)
        hint = explanation_data.get('hint', 'No hint available')
        solution = explanation_data.get('solution', 'No solution available')
        return hint, solution
    except json.JSONDecodeError:
        return 'No hint available', 'No solution available'

# Function to generate misconceptions-based incorrect options

def generate_incorrect_options(row):

    chain = LLMChain(llm=llm, prompt=prompt_template)
    try:
        response = chain.run(
            system_prompt=DISTRACTOR_SYSTEM_PROMPT,
            topic=row['topic'],
            topic_id=row['topic_id'],
            question=row['question'],
            hint=row['hint'],
            solution=row['solution']
        )
        # if response.strip().startswith("```json"):
        #     response = response.strip().strip("```json").strip("```").strip()
        return response
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return None

# Apply the function to generate misconceptions-based incorrect options for each question
misconception_options = []
def loopForDistractor(data):
    distractor_counter = 0
    retries_counter = 0
    maxCounter = len(data)
    while distractor_counter<maxCounter:
        row = data.loc[distractor_counter]
        print(f"Processing question {distractor_counter + 1} out of {len(data)}")
        misconception_option = generate_incorrect_options(row)
        parsed_response = parse_Response(misconception_option)

        if parsed_response:
            # misconception_options.append(parsed_response)
            # row['misconception_options'] = parsed_response
            # print(distractor_counter)
            # print(data.loc[distractor_counter , 'misconception_options'])
            # print(parsed_response)

            data.loc[distractor_counter , 'misconception_options'] =[parsed_response]
            distractor_counter += 1
            retries_counter = 0
        else:
            print(f"Failed to parse response or generate distractors for question {distractor_counter + 1}, retrying...")
            retries_counter += 1
        if retries_counter == 3:
            # misconception_option.append("")
            data.loc[distractor_counter , 'misconception_options'] = ""
            distractor_counter += 1

loopForDistractor(data)
# data['misconception_options'] = misconception_options

Processing question 1 out of 121
Processing question 2 out of 121
Processing question 3 out of 121
Processing question 4 out of 121
Processing question 5 out of 121
Processing question 6 out of 121
Processing question 7 out of 121
Processing question 8 out of 121
Processing question 9 out of 121
Processing question 10 out of 121
Processing question 11 out of 121
Processing question 12 out of 121
Processing question 13 out of 121
Processing question 14 out of 121
Processing question 15 out of 121
Processing question 16 out of 121
Processing question 17 out of 121
Processing question 18 out of 121
Processing question 19 out of 121
Processing question 20 out of 121
Processing question 21 out of 121
Processing question 22 out of 121
Processing question 23 out of 121
Processing question 24 out of 121
Processing question 25 out of 121
Processing question 26 out of 121
Processing question 27 out of 121
Processing question 28 out of 121
Processing question 29 out of 121
Processing question 30 

In [28]:
for i in range(len(data)):
    print(data['misconception_options'][i])

[{'correct_option': '\\begin{aligned} &\\quad 3.84 \\mathrm{~m}^3 \\end{aligned}', 'option1': {'option': '\\begin{aligned} &\\quad 3.84 \\mathrm{~cm}^3 \\end{aligned}', 'rationale': '\\begin{aligned} &\\quad \\text{Confused the units by not converting all dimensions to meters before calculating volume.} \\end{aligned}'}, 'option2': {'option': '\\begin{aligned} &\\quad 38.4 \\mathrm{~m}^3 \\end{aligned}', 'rationale': '\\begin{aligned} &\\quad \\text{Forgot to correctly multiply the dimensions, leading to an incorrect volume calculation.} \\end{aligned}'}, 'option3': {'option': '\\begin{aligned} &\\quad 3.2 \\mathrm{~m}^3 \\end{aligned}', 'rationale': '\\begin{aligned} &\\quad \\text{Ignored converting the height from centimeters to meters, using incorrect dimensions.} \\end{aligned}'}}]
[{'correct_option': '\\begin{aligned} & \\text{1200} \\ \\mathrm{cm}^{3} \\ \\text{or} \\ 0.0012 \\ \\mathrm{m}^{3} \\end{aligned}', 'option1': {'option': '\\begin{aligned} & \\text{600} \\ \\mathrm{cm}

### Output Formatting & Pushing into the DB

In [28]:
# Function to process each row and extract required fields
def process_row(row, index):
    try:
        misconception_data = row["misconception_options"]
        # Extract topic, question, and solution details
        topic_id = row["topic_id"]
        topic_name = row["topic"]
        question = row["question"]
        hint = row["hint"]
        explanation = row["solution"]
        # correct_answer = row["final_answer"]
        chapter = conf_data['chapter_name']

        # Extract option and rationale details
        options = {}
        for i in range(1, 4):  # Assuming there are 4 options for each question
            options["correct_answer"] = misconception_data[0]["correct_option"]
            option_key = f'option{i}'
            option_info = misconception_data[0][option_key]
            options[f'option{i}'] = option_info.get('option', '')
            options[f'dr{i}'] = option_info.get('rationale', '')
        
        # Construct the final JSON structure
        start = "\\begin{aligned}"
        end = "\\end{aligned}"
        
        combined_data = {
            'topic_id': topic_id,
            'topic_name': topic_name,
            'grade' : conf_data['grade'],
            'board' : "ICSE",
            
            'subject' : subject,
            'chapter_name' : chapter,
            'publication' : conf_data['publication'],
            'question': [{"content": question}],
            # if (str(question).startswith("\\begin{aligned}") and str(question).endswith("\\end{aligned}")) else [{"content": start + question + end}],

            'hint': [{"content": hint}],
            # if (str(hint).startswith("\\begin{aligned}") and str(hint).endswith("\\end{aligned}")) else [{"content": start + hint + end}],

            'solution': [{"content": explanation}],
            # if (str(explanation).startswith("\\begin{aligned}") and str(explanation).endswith("\\end{aligned}")) else [{"content": start + explanation + end}],

            'final_answer': [{"content": options.get('correct_answer', '')}],
            # if (str(options.get('correct_answer', '')).startswith("\\begin{aligned}") and str(options.get('correct_answer', '')).endswith("\\end{aligned}")) else [{"content": start + options.get('correct_answer', '') + end}],

            'option1': [{"content": options.get('option1', '')}],
            # if (str(options.get('option1')).startswith("\\begin{aligned}") and str(options.get('option1')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option1', '')) + end} ],
            'dr1': [{"content": options.get('dr1', '')}],
            'option2': [{"content": options.get('option2', '')}],
            # if (str(options.get('option2')).startswith("\\begin{aligned}") and str(options.get('option2')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option2', '')) + end}],
            'dr2': [{"content": options.get('dr2', '')}],
            'option3': [{"content": options.get('option3', '')}],
            # if (str(options.get('option3')).startswith("\\begin{aligned}") and str(options.get('option3')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option3', '')) + end}],
            'dr3': [{"content": options.get('dr3', '')}]


        }

        return combined_data
    except Exception as e:
        # If there is an error, return an empty structure with an error message
        return {
            'topic_id': row.get('topic_id', f"Topic {index + 1}"),
            'error': f"Exception Occurred: {type(e).__name__} – {e} \n {traceback.format_exc()}"
        }

# Process all rows in the dataframe
final_output = [process_row(row, idx) for idx, row in data.iterrows()]
final_output[2]

{'topic_id': '6_physics_physical quantities and measurement_1',
 'topic_name': 'Measurement of Length',
 'grade': '6',
 'board': 'ICSE',
 'subject': 'physics',
 'chapter_name': 'Physical Quantities and Measurement',
 'publication': 'goyal',
 'question': [{'content': '\\begin{aligned} &\\text{Why is it important to have standard units for measurement?} \\end{aligned}'}],
 'hint': [{'content': '\\begin{aligned} &\\text{Consider how having a common standard for measurement helps in ensuring consistency and accuracy across different regions and fields.} \\\\ &\\text{Think about the difficulties that might arise if everyone used different units for the same quantity.} \\\\ \\end{aligned}'}],
 'solution': [{'content': '\\begin{aligned} &\\text{Standard units of measurement are crucial because they provide a consistent framework for communication and comparison.} \\\\ &\\text{Without standard units, it would be challenging to compare measurements accurately, as different regions might use dif

In [29]:
final_output

[{'topic_id': '6_physics_physical quantities and measurement_1',
  'topic_name': 'Measurement of Length',
  'grade': '6',
  'board': 'ICSE',
  'subject': 'physics',
  'chapter_name': 'Physical Quantities and Measurement',
  'publication': 'goyal',
  'question': [{'content': '\\begin{aligned} &\\text{If a thin copper wire is wound around a pencil to form a coil with 25 turns, and the length of the coil is measured to be 2 cm, what is the diameter of the wire?} \\end{aligned}'}],
  'hint': [{'content': '\\begin{aligned} &\\text{To find the diameter of the wire, consider the total length of the coil and the number of turns.} \\\\ &\\text{The length of the coil is the product of the number of turns and the circumference of the wire.} \\\\ &\\text{Use the formula for circumference to find the diameter.} \\\\ \\end{aligned}'}],
  'solution': [{'content': '\\begin{aligned} &\\text{The length of the coil is given as 2 cm, and there are 25 turns.} \\\\ &\\text{The length of the coil is equal to

#### Latex Formatting to store in DB

In [30]:
from config import MONGO_URL , DB_NAME

In [31]:
dbCollection = "LatexTest"
client = AsyncIOMotorClient(MONGO_URL)

In [32]:
async def pushToDB(dbContent, subject):
    try:
        db = client[DB_NAME]
        que_collection = db[dbCollection]

        dbContent["status"] = "not_reviewed"
        dbContent["comment"] = ""
        dbContent["testFlag"] = "true"
        dbContent["subject"] = subject
        await que_collection.insert_one(dbContent)
        print("Data Uploaded to the database successfully.")
    except Exception as e:
        print("Exception Occurred: ", type(e).__name__, "–", e, "\n", traceback.format_exc())

In [33]:
for doc in final_output:
    await pushToDB(doc, subject)
client.close()

Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database su